# ETL: Entregable 1

In [92]:
from dotenv import dotenv_values
from sqlalchemy import create_engine
from sqlalchemy import text
import datetime
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyOAuth

In [93]:
def check_if_valid_data(df: pd.DataFrame) -> bool:
    # Check if dataframe is empty
    if df.empty:
        print("No songs downloaded. Finishing execution")
        return False
    
    # Primary Key check
    if not df["played_at"].is_unique:
        raise Exception("A value from played_at is not unique")
    
    # Check for nulls
    if df.isnull().values.any():
        raise Exception("A value in df is null")
        
    return True

In [94]:
config = dotenv_values(".env")

In [95]:
# Spotifg settings
CLIENT_ID = config["CLIENT_ID"]
CLIENT_SECRET = config["CLIENT_SECRET"]
SPOTIPY_REDIRECT_URI = config["SPOTIPY_REDIRECT_URI"]
SCOPE = config["SCOPE"]

In [96]:
# DB settings
TABLENAME = config["TABLENAME"]
DB_USERNAME = config["DB_USERNAME"]
DB_PASSWORD = config["DB_PASSWORD"]
DB_HOST = config["DB_HOST"]
DB_PORT = config["DB_PORT"]
DB_NAME = config["DB_NAME"]
DB_SCHEMA = config["DB_SCHEMA"]

In [97]:
# Connect to Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=SPOTIPY_REDIRECT_URI,
                                               scope=SCOPE))

# Convert a unix timestamp in milliseconds
today = datetime.datetime.now()
yesterday = today - datetime.timedelta(days=1)
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
yesterday_unix_timestamp = int(yesterday.timestamp()) * 1000
limit = 50    

# Retorna un diccionario con las canciones escuchadas
raw_data = sp.current_user_recently_played(limit=limit, after=yesterday_unix_timestamp)

In [98]:
raw_data.keys()

dict_keys(['items', 'next', 'cursors', 'limit', 'href'])

In [99]:
song_names = []
artist_names = []
played_at_list = []
timestamps = []

In [100]:
for song in raw_data["items"]:
    song_names.append(song["track"]["name"])
    artist_names.append(song["track"]["artists"][0]["name"])
    played_at_list.append(song["played_at"]),
    timestamps.append(song["played_at"][0:10])

In [101]:
song_dict = {
    "song_name" : song_names,
    "artist_name": artist_names,
    "played_at" : played_at_list,
    "timestamp" : timestamps
}

In [75]:
song_df = pd.DataFrame(song_dict, columns = ["song_name",
                                             "artist_name",
                                             "played_at",
                                             "timestamp"])

In [102]:
song_df.head(5)

,song_name,artist_name,played_at,timestamp
5,These Days,Foo Fighters,2023-05-25T19:20:52.062Z,2023-05-25
6,These Days,Foo Fighters,2023-05-25T18:12:17.556Z,2023-05-25
7,Baby Queen,Gorillaz,2023-05-25T18:03:50.259Z,2023-05-25
8,New Gold (feat. Tame Impala and Bootie Brown),Gorillaz,2023-05-25T18:00:10.075Z,2023-05-25
9,Azul pálido,Cascarón de Proa,2023-05-25T17:56:06.589Z,2023-05-25


In [103]:
# eliminar fechas diferentes a las que queremos
clean_song_df = song_df[pd.to_datetime(song_df["played_at"]).dt.date == yesterday.date()]

In [104]:
clean_song_df

,song_name,artist_name,played_at,timestamp
5,These Days,Foo Fighters,2023-05-25T19:20:52.062Z,2023-05-25
6,These Days,Foo Fighters,2023-05-25T18:12:17.556Z,2023-05-25
7,Baby Queen,Gorillaz,2023-05-25T18:03:50.259Z,2023-05-25
8,New Gold (feat. Tame Impala and Bootie Brown),Gorillaz,2023-05-25T18:00:10.075Z,2023-05-25
9,Azul pálido,Cascarón de Proa,2023-05-25T17:56:06.589Z,2023-05-25


In [105]:
# Data validation
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")

Data valid, proceed to Load stage...


In [88]:
# Load

rows_imported = 0

conn_string = f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)
conn = engine.connect()

sql_query = text(
"""
CREATE TABLE IF NOT EXISTS coder.my_tracks_history(
    id_my_tracks_history SERIAL NOT NULL,
    song_name VARCHAR(200) NOT NULL,
    artist_name VARCHAR(200) NOT NULL,
    played_at TIMESTAMP NOT NULL,
    timestamp DATE NOT NULL,

    CONSTRAINT pk_my_tracks_history PRIMARY KEY (played_at)
);
""")

conn.execute(sql_query)
conn.commit()

print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {TABLENAME}")

clean_song_df.to_sql(TABLENAME,
               con=engine,
               schema=DB_SCHEMA,
               index=False,
               if_exists="append")

rows_imported += clean_song_df.shape[0]
print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {TABLENAME}")
print(f"Data imported successful")

conn.close()

Importing rows 0 to 5... for table my_tracks_history
Importing rows 5 to 5... for table my_tracks_history
Data imported successful
